In [ ]:
import os
import re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("MatricaDataPipeline").getOrCreate()

# Dynamically find the project root regardless of who runs it
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
BRONZE_DIR = os.path.join(DATA_DIR, "bronze")
SILVER_DIR = os.path.join(DATA_DIR, "silver")
GOLD_DIR = os.path.join(DATA_DIR, "gold")

def quality_report(df):
    total = df.count()
    print(f"Total Rows: {total}")
    null_counts = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns[:10]]
    if null_counts:
        df.select(null_counts).show()

def standardize_columns(df):
    for c in df.columns:
        new_c = str(c).strip().replace(' ', '_')
        new_c = re.sub(r'[^a-zA-Z0-9_]', '', new_c).lower()
        if c != new_c:
            df = df.withColumnRenamed(c, new_c)
    return df

def trim_string_columns(df):
    for field in df.schema.fields:
        if field.dataType.typeName() == 'string':
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
    return df

def blank_to_null(df):
    for field in df.schema.fields:
        if field.dataType.typeName() == 'string':
            df = df.withColumn(field.name, F.when(F.col(field.name) == "", None).otherwise(F.col(field.name)))
    return df

def standardize_text(df):
    return df

def remove_duplicates(df):
    return df.dropDuplicates()

def fill_numeric_nulls(df):
    return df.na.fill(0)

def remove_empty_rows(df):
    return df.dropna(how="all")

def write_silver(df, folder_name, dataset_name):
    out_path = os.path.join(SILVER_DIR, folder_name, dataset_name)
    df.write.mode("overwrite").parquet(out_path)

def validate_silver(folder_name, dataset_name):
    out_path = os.path.join(SILVER_DIR, folder_name, dataset_name)
    df = spark.read.parquet(out_path)
    print(f"Silver validation successful: {df.count()} rows found in {out_path}")


In [ ]:
# ==============================================================================
# Dataset : draft_phase
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate draft_phase
# ==============================================================================

print("=" * 80)
print("Processing Dataset : draft_phase")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "draft_phase"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : eco_rounds
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate eco_rounds
# ==============================================================================

print("=" * 80)
print("Processing Dataset : eco_rounds")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "eco_rounds"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : eco_stats
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate eco_stats
# ==============================================================================

print("=" * 80)
print("Processing Dataset : eco_stats")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "eco_stats"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : kills
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate kills
# ==============================================================================

print("=" * 80)
print("Processing Dataset : kills")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "kills"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : kills_stats
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate kills_stats
# ==============================================================================

print("=" * 80)
print("Processing Dataset : kills_stats")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "kills_stats"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = ['2k', '3k', '4k', '5k', '1v1', '1v2', '1v3', '1v4', '1v5', 'econ', 'spike_plants', 'spike_defuses']
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : maps_played
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate maps_played
# ==============================================================================

print("=" * 80)
print("Processing Dataset : maps_played")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "maps_played"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : maps_scores
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate maps_scores
# ==============================================================================

print("=" * 80)
print("Processing Dataset : maps_scores")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "maps_scores"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : overview
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate overview
# ==============================================================================

print("=" * 80)
print("Processing Dataset : overview")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "overview"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = ['kills', 'deaths', 'assists']
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : rounds_kills
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate rounds_kills
# ==============================================================================

print("=" * 80)
print("Processing Dataset : rounds_kills")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "rounds_kills"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : scores
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate scores
# ==============================================================================

print("=" * 80)
print("Processing Dataset : scores")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "scores"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : team_mapping
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate team_mapping
# ==============================================================================

print("=" * 80)
print("Processing Dataset : team_mapping")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "team_mapping"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : win_loss_methods_count
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate win_loss_methods_count
# ==============================================================================

print("=" * 80)
print("Processing Dataset : win_loss_methods_count")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "win_loss_methods_count"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)


In [ ]:
# ==============================================================================
# Dataset : win_loss_methods_round_number
# Layer   : Bronze -> Silver
# Purpose : Clean & Validate win_loss_methods_round_number
# ==============================================================================

print("=" * 80)
print("Processing Dataset : win_loss_methods_round_number")
print("=" * 80)


In [ ]:
# ------------------------------------------------------------------------------
# Step 1 : Read Bronze Dataset
# ------------------------------------------------------------------------------
dataset_name = "win_loss_methods_round_number"
folder_name = "matches"
df = spark.read.csv(os.path.join(BRONZE_DIR, folder_name, f"{dataset_name}.csv"), header=True, inferSchema=True)
rows_before = df.count()
print(f"Rows Read : {rows_before}")
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 2 : Data Quality Report (Before)
# ------------------------------------------------------------------------------
print("\nData Quality Report (Before Cleaning)")
quality_report(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 3 : Standardize Column Names
# ------------------------------------------------------------------------------
df = standardize_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 4 : Trim String Columns
# ------------------------------------------------------------------------------
df = trim_string_columns(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 5 : Replace Blank Strings with NULL
# ------------------------------------------------------------------------------
df = blank_to_null(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 6 : Standardize Text
# ------------------------------------------------------------------------------
df = standardize_text(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 7 : Remove Duplicate Rows
# ------------------------------------------------------------------------------
before_duplicates = df.count()
df = remove_duplicates(df)
after_duplicates = df.count()
duplicates_removed = before_duplicates - after_duplicates


In [ ]:
# ------------------------------------------------------------------------------
# Step 8 : Fill Numeric NULL Values
# ------------------------------------------------------------------------------
df = fill_numeric_nulls(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 9 : Validate Numeric Columns
# ------------------------------------------------------------------------------
numeric_columns = []
invalid_records = None
for column in numeric_columns:
    condition = F.col(f"`{column}`") < 0
    if invalid_records is None:
        invalid_records = condition
    else:
        invalid_records = invalid_records | condition

if numeric_columns:
    invalid_count = df.filter(invalid_records).count()
    print(f"Invalid Numeric Records : {invalid_count}")
    df = df.filter(~invalid_records)
else:
    invalid_count = 0


In [ ]:
# ------------------------------------------------------------------------------
# Step 11 : Remove Empty Rows
# ------------------------------------------------------------------------------
df = remove_empty_rows(df)


In [ ]:
# ------------------------------------------------------------------------------
# Step 12 : Data Quality Report (After)
# ------------------------------------------------------------------------------
print("\nData Quality Report (After Cleaning)")
quality_report(df)
df.show(5)


In [ ]:
# ------------------------------------------------------------------------------
# Step 13 : Write Silver Dataset
# ------------------------------------------------------------------------------
write_silver(df, folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 14 : Validate Silver Dataset
# ------------------------------------------------------------------------------
validate_silver(folder_name, dataset_name)


In [ ]:
# ------------------------------------------------------------------------------
# Step 15 : ETL Summary
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ETL EXECUTION SUMMARY")
print("=" * 80)
print(f"Dataset              : {dataset_name}")
print(f"Rows Read            : {rows_before}")
print(f"Rows Written         : {df.count()}")
print(f"Duplicates Removed   : {duplicates_removed}")
print(f"Invalid Records      : {invalid_count}")
print(f"Columns              : {len(df.columns)}")
print(f"Silver Location      : {SILVER_DIR}\{folder_name}\{dataset_name}")
print("Status               : SUCCESS")
print("=" * 80)
